# Age prediction model
This notebook explains the creation and training of a predictive model to be used in an app to guess peoples' ages based in questions.

In [1]:
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

## Data cleaning (Human_Age_Prediction.csv)
use the csv to create a dataframe ready to feed the model into

In [2]:
# human age prediction, cleaned dataframe

# filtered and normalized columns title
df_human = pd.read_csv("../data/Human_Age_Prediction.csv", usecols=["Bone Density (g/cm²)", "Vision Sharpness", "Hearing Ability (dB)", "Physical Activity Level", "Smoking Status", "Alcohol Consumption", "Diet",  "Mental Health Status", "Sleep Patterns", "Stress Levels", "Income Level", "Age (years)"])
df_human.columns = ["bone_density", "vision_sharpness", "hearing_ability", "physical_activity_level", "smoking_status", "alcohol_consumption", "diet", "mental_health_status", "sleep_patterns", "stress_levels", "income_level", "age"]


# ordinal encoding for physical_activity_level, income_level, mental-health_status, alcohol conmusion
ordinal_map = {
    'physical_activity_level': {'Low': 0, 'Moderate': 1, 'High': 2},
    'alcohol_consumption': {'None': 0, 'Occasional': 1, 'Frequent': 2},
    'mental_health_status': {'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3},
    'income_level': {'Low': 0, 'Medium': 1, 'High': 2},
}

for col, mapping in ordinal_map.items():
    df_human[col] = df_human[col].map(mapping)


# label encoding for sleep_patterns and diet
nominal_cols = ['smoking_status', 'diet', 'sleep_patterns']

# encoding using label encoder
label_encoders = {}
for col in nominal_cols:
    df_human[col] = df_human[col].astype('object')
    mask = df_human[col].notna()
    label_encoders[col] = LabelEncoder()
    df_human.loc[mask, col] = label_encoders[col].fit_transform(df_human.loc[mask, col])
    df_human[col] = df_human[col].fillna(-1).astype(int)
    
# print(df_human.head(2))
print(df_human.dtypes)

bone_density               float64
vision_sharpness           float64
hearing_ability            float64
physical_activity_level      int64
smoking_status               int64
alcohol_consumption        float64
diet                         int64
mental_health_status         int64
sleep_patterns               int64
stress_levels              float64
income_level                 int64
age                          int64
dtype: object


## Data cleaning (Wellbeing_and_lifestyle.csv)
not currently in use

In [3]:
# not in use for now

# Wellbeing data, cleaned dataframe. 
df_wellbeing = pd.read_csv("../data/Wellbeing_and_lifestyle_data_Kaggle.csv", usecols=["FRUITS_VEGGIES", "DAILY_STRESS", "PLACES_VISITED", "CORE_CIRCLE", "DAILY_STEPS", "SLEEP_HOURS", "AGE"])
df_wellbeing.columns = ["wb_fruits_veggies", "wb_daily_stress", "wb_places_visited", "wb_core_cirle", "wb_daily_steps", "wb_sleep_hours", "age"]

# convert daily_stress to int64 type
df_wellbeing['wb_daily_stress'] = pd.to_numeric(df_wellbeing['wb_daily_stress'], errors="coerce")
df_wellbeing = df_wellbeing.dropna(subset=["wb_daily_stress"])
df_wellbeing["wb_daily_stress"] = df_wellbeing["wb_daily_stress"].astype(int)

# convert age to int


# print(df_wellbeing.head(2))
print(df_wellbeing.dtypes)

wb_fruits_veggies    int64
wb_daily_stress      int64
wb_places_visited    int64
wb_core_cirle        int64
wb_daily_steps       int64
wb_sleep_hours       int64
age                    str
dtype: object


## Model training
assigning X and y to the test split

In [4]:
# train / test split
X = df_human.drop('age', axis=1)
y = df_human['age']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Define model and fit

In [5]:
# define XGB regressor model and fit 
model = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1)
model.fit(X_train, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


## model testing and evaluation

In [6]:
# evaluate the model
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"MAE: {mae}")
print(f"R²: {r2}")

MAE: 4.789032459259033
R²: 0.9088882207870483


In [7]:
# finding age range
smallest_age = df_human['age'].min()
biggest_age = df_human['age'].max()
mean = df_human['age'].mean()
median = df_human['age'].median()

print(f"range: {smallest_age} - {biggest_age} \nmean: {mean}\nmedian: {median}")

range: 18 - 89 
mean: 53.48566666666667
median: 53.0


In [8]:
# age distribution
fig = px.histogram(df_human, x='age', title='Age distribution')
fig.show()

In [9]:
# MAE by age group

# separate dataframe in bins
bins = pd.cut(y_test, bins=[17, 30, 45, 60, 90], labels=['Young', 'Middle', 'Senior', 'Elder'])

# store results in a dataframe to compare
results = pd.DataFrame({
    'actual': y_test,
    'predicted': predictions,
    'age_group': bins
})
#calculate absolute error
results['abs_error'] = (results['actual'] - results['predicted']).abs()

# calculate mae per group
mae_by_group = results.groupby('age_group')['abs_error'].mean()
print(mae_by_group)

# create a df for the mae
mae_df = mae_by_group.reset_index()
mae_df.columns = ['age_group', 'mae']

fig = px.bar(mae_df, x='age_group', y='mae')
fig.show()


age_group
Young     3.490937
Middle    5.397493
Senior    5.588566
Elder     4.517482
Name: abs_error, dtype: float64
